# Car Price Prediction with MLflow

In [26]:
import json
from pathlib import Path

import joblib
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [27]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'carprice.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'carprice.csv'
MODEL_DIR = PROJECT_ROOT / 'models'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
EXPERIMENT_NAME = 'CarPricePrediction'
TARGET_COLUMN = 'selling_price'

In [28]:
df = pd.read_csv(DATA_PATH)
df.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage(km/ltr/kg),engine,max_power,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.40,1248.0,74,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14,1498.0,103.52,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.70,1497.0,78,5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.00,1396.0,90,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.10,1298.0,88.2,5.0


In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8128 entries, 0 to 8127
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   name                8128 non-null   object 
 1   year                8128 non-null   int64  
 2   selling_price       8128 non-null   int64  
 3   km_driven           8128 non-null   int64  
 4   fuel                8128 non-null   object 
 5   seller_type         8128 non-null   object 
 6   transmission        8128 non-null   object 
 7   owner               8128 non-null   object 
 8   mileage(km/ltr/kg)  7907 non-null   float64
 9   engine              7907 non-null   float64
 10  max_power           7913 non-null   object 
 11  seats               7907 non-null   float64
dtypes: float64(3), int64(3), object(6)
memory usage: 762.1+ KB


In [30]:
for column in ['mileage(km/ltr/kg)', 'engine', 'max_power', 'seats']:
    df[column] = pd.to_numeric(df[column], errors='coerce')

df['name'] = df['name'].astype(str).str.strip()
df = df.dropna(subset=[TARGET_COLUMN])
df.isna().sum()

name                    0
year                    0
selling_price           0
km_driven               0
fuel                    0
seller_type             0
transmission            0
owner                   0
mileage(km/ltr/kg)    221
engine                221
max_power             216
seats                 221
dtype: int64

In [31]:
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_features = ['year', 'km_driven', 'mileage(km/ltr/kg)', 'engine', 'max_power', 'seats']
categorical_features = ['name', 'fuel', 'seller_type', 'transmission', 'owner']

In [32]:
def build_pipeline(model):
    numeric_transformer = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features),
        ]
    )

    return Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('model', model),
        ]
    )


def evaluate(model, x_test, y_test):
    predictions = model.predict(x_test)
    mse = mean_squared_error(y_test, predictions)
    return {
        'rmse': float(mse ** 0.5),
        'mae': float(mean_absolute_error(y_test, predictions)),
        'r2': float(r2_score(y_test, predictions)),
    }

In [33]:
mlflow.set_experiment(EXPERIMENT_NAME)

candidate_models = {
    'linear_regression': LinearRegression(),
    'random_forest': RandomForestRegressor(
        n_estimators=250,
        max_depth=18,
        min_samples_split=4,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    ),
}

best_model = None
best_metrics = None
best_run_id = None

for model_name, estimator in candidate_models.items():
    pipeline = build_pipeline(estimator)
    with mlflow.start_run(run_name=model_name):
        pipeline.fit(X_train, y_train)
        metrics = evaluate(pipeline, X_test, y_test)

        mlflow.log_param('model_name', model_name)
        mlflow.log_param('train_rows', len(X_train))
        mlflow.log_param('test_rows', len(X_test))
        mlflow.log_param('feature_count', X.shape[1])

        for metric_name, metric_value in metrics.items():
            mlflow.log_metric(metric_name, metric_value)

        mlflow.sklearn.log_model(
            sk_model=pipeline,
            artifact_path='model',
            input_example=X_test.head(2),
        )

        if best_metrics is None or metrics['r2'] > best_metrics['r2']:
            best_model = pipeline
            best_metrics = metrics | {'model_name': model_name}
            best_run_id = mlflow.active_run().info.run_id

best_metrics

2026/03/13 23:00:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 23:00:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
d:\MLOPS\CAR_PRICE\venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you ca

{'rmse': 137773.91228195795,
 'mae': 69318.65401940442,
 'r2': 0.9710417978895759,
 'model_name': 'random_forest'}

In [34]:
MODEL_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR.mkdir(exist_ok=True)
model_path = MODEL_DIR / 'model.pkl'
metadata_path = ARTIFACT_DIR / 'model_metadata.json'

joblib.dump(best_model, model_path)
metadata = {
    'experiment_name': EXPERIMENT_NAME,
    'best_model_name': best_metrics['model_name'],
    'best_run_id': best_run_id,
    'metrics': best_metrics,
    'data_path': str(DATA_PATH),
    'model_path': str(model_path),
}

metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
metadata

{'experiment_name': 'CarPricePrediction',
 'best_model_name': 'random_forest',
 'best_run_id': '86cdefa8c23e49918754f2fe40e07bfd',
 'metrics': {'rmse': 137773.91228195795,
  'mae': 69318.65401940442,
  'r2': 0.9710417978895759,
  'model_name': 'random_forest'},
 'data_path': 'd:\\MLOPS\\CAR_PRICE\\data\\carprice.csv',
 'model_path': 'd:\\MLOPS\\CAR_PRICE\\models\\model.pkl'}